# Questão 2 – Comparação de Classificadores no Conjunto Ionosphere

**Dataset**: Ionosphere (UCI) — 351 amostras, 33 features (removida coluna constante)  
**Versões**:
- **V1**: labels originais binários ('g'=1, 'b'=0)  
- **V2**: labels de cluster do KCM-K-GH (c*=5, da Questão 1)

**Protocolo**: 30 repetições de 10-fold CV estratificado (externo), 5-fold interno para seleção de hiperparâmetros

**Classificadores**:
1. Gaussiano Bayesiano (MLE, normal multivariada com Σ distinta por classe)
2. kNN Bayesiano (distâncias Euclidiana, City-Block, Chebyshev; tunar k e distância)
3. Janela de Parzen (kernel produto Gaussiano univariado; tunar h)
4. Regressão Logística (tunar C)
5. Voto Majoritário (combinação de i–iv)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.special import logsumexp
from scipy.stats import t as t_dist, f as f_dist
from itertools import product as iterproduct
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')


In [ ]:
# ── Carregamento do dataset ──────────────────────────────────────────────
ionosphere = fetch_openml(name='ionosphere', version=1, as_frame=True)
X_raw = ionosphere.data.values.astype(float)
y_raw = ionosphere.target.values

# Remover coluna constante (coluna 1, zeros)
col_std = X_raw.std(axis=0)
non_const = col_std > 1e-10
X = X_raw[:, non_const]          # (351, 33)
y_v1 = (y_raw == 'g').astype(int)  # V1: binário

print(f"X shape: {X.shape}")
print(f"V1 – classes: {np.bincount(y_v1)}  (0=bad, 1=good)")


In [ ]:
# ── Labels V2: carregar de Q1 ou recomputar ─────────────────────────────
def compute_kernel(X, g, inv_s2):
    N, c = X.shape[0], len(g)
    K = np.zeros((N, c))
    for k in range(c):
        diff2 = (X - g[k]) ** 2
        K[:, k] = np.exp(-0.5 * (diff2 @ inv_s2))
    return K

def kcm_k_gh(X, c, max_iter=300, eps=1e-10, seed=None):
    rng = np.random.default_rng(seed)
    N, P = X.shape
    idx = rng.choice(N, c, replace=False)
    g = X[idx].copy()
    inv_s2 = np.ones(P)
    K = compute_kernel(X, g, inv_s2)
    labels = np.argmax(K, axis=1)
    J_hist = []
    for _ in range(max_iter):
        old_labels = labels.copy()
        # Step 1: protótipos
        K = compute_kernel(X, g, inv_s2)
        for k in range(c):
            mask = labels == k
            if mask.sum() > 0:
                w = K[mask, k]
                g[k] = (X[mask].T @ w) / (w.sum() + eps)
        # Step 2: larguras
        K = compute_kernel(X, g, inv_s2)
        D = np.zeros(P)
        for k in range(c):
            mask = labels == k
            if mask.sum() > 0:
                w = K[mask, k]
                diff2 = (X[mask] - g[k]) ** 2
                D += diff2.T @ w
        D_safe = np.maximum(D, eps)
        inv_s2 = np.exp(np.mean(np.log(D_safe)) - np.log(D_safe))
        # Step 3: alocação
        K = compute_kernel(X, g, inv_s2)
        labels = np.argmax(K, axis=1)
        J = 2.0 * np.sum(1.0 - K[np.arange(N), labels])
        J_hist.append(J)
        if np.array_equal(labels, old_labels):
            break
    return g, 1.0/inv_s2, labels, J_hist

if os.path.exists('labels_opt.npy'):
    y_v2 = np.load('labels_opt.npy')
    print(f"labels_opt.npy carregado. Shape: {y_v2.shape}")
else:
    print("labels_opt.npy não encontrado. Recomputando KCM-K-GH c*=5 (100 runs)...")
    best_J = np.inf
    best_labels = None
    for run in range(100):
        _, _, labels, J_hist = kcm_k_gh(X, c=5, seed=run)
        if J_hist[-1] < best_J:
            best_J = J_hist[-1]
            best_labels = labels.copy()
    y_v2 = best_labels
    np.save('labels_opt.npy', y_v2)
    print(f"Computado. J={best_J:.4f}")

print(f"V2 – distribuição: {np.bincount(y_v2)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CLASSIFICADORES
# ═══════════════════════════════════════════════════════════════════════════

class GaussianBayesClassifier:
    """MLE Gaussiano Bayesiano – normal multivariada com Σ distinta por classe (QDA)"""
    def __init__(self, reg=1e-6):
        self.reg = reg

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        N, P = X.shape
        self.priors_ = {}; self.means_ = {}
        self.inv_covs_ = {}; self.log_dets_ = {}
        for c in self.classes_:
            Xc = X[y == c]
            self.priors_[c] = len(Xc) / N
            self.means_[c] = Xc.mean(axis=0)
            cov = np.cov(Xc.T) + self.reg * np.eye(P)
            sign, logdet = np.linalg.slogdet(cov)
            self.log_dets_[c] = logdet
            self.inv_covs_[c] = np.linalg.inv(cov)
        return self

    def predict(self, X):
        log_posts = np.zeros((len(X), len(self.classes_)))
        for i, c in enumerate(self.classes_):
            diff = X - self.means_[c]
            maha = np.sum(diff @ self.inv_covs_[c] * diff, axis=1)
            log_posts[:, i] = -0.5 * (maha + self.log_dets_[c]) + np.log(self.priors_[c])
        return self.classes_[np.argmax(log_posts, axis=1)]

    def get_params(self, deep=True): return {'reg': self.reg}
    def set_params(self, **p):
        for k,v in p.items(): setattr(self,k,v)
        return self


class ParzenWindowClassifier:
    """Janela de Parzen – kernel produto de Gaussianas univariadas"""
    def __init__(self, h=1.0):
        self.h = h

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_by_class_ = {}; self.log_priors_ = {}
        N = len(X)
        for c in self.classes_:
            mask = y == c
            self.X_by_class_[c] = X[mask]
            self.log_priors_[c] = np.log(mask.sum() / N)
        return self

    def _log_density(self, X_q, Xc):
        """Vectorized log-density for all query points"""
        h, P = self.h, Xc.shape[1]
        diff = (X_q[:, np.newaxis, :] - Xc[np.newaxis, :, :]) / h  # (Nq, Nc, P)
        log_k = -0.5 * np.sum(diff**2, axis=2) - P * np.log(h * np.sqrt(2*np.pi))  # (Nq, Nc)
        return logsumexp(log_k, axis=1) - np.log(len(Xc))  # (Nq,)

    def predict(self, X):
        log_posts = np.zeros((len(X), len(self.classes_)))
        for i, c in enumerate(self.classes_):
            log_posts[:, i] = self._log_density(X, self.X_by_class_[c]) + self.log_priors_[c]
        return self.classes_[np.argmax(log_posts, axis=1)]

    def get_params(self, deep=True): return {'h': self.h}
    def set_params(self, **p):
        for k,v in p.items(): setattr(self,k,v)
        return self


class MajorityVoteClassifier:
    """Regra da maioria (Kittler et al. 1998, Eq.20)"""
    def __init__(self, classifiers):
        self.classifiers = classifiers

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        for clf in self.classifiers: clf.fit(X, y)
        return self

    def predict(self, X):
        preds = np.array([clf.predict(X) for clf in self.classifiers])  # (k, N)
        result = np.zeros(len(X), dtype=preds.dtype)
        for i in range(len(X)):
            vals, counts = np.unique(preds[:, i], return_counts=True)
            result[i] = vals[np.argmax(counts)]
        return result


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FUNÇÕES DE TUNING (inner 5-fold CV)
# ═══════════════════════════════════════════════════════════════════════════

def _cv_score(clf, X, y, n_folds, avg):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=7)
    scores = []
    for tr, val in skf.split(X, y):
        clf.fit(X[tr], y[tr])
        scores.append(f1_score(y[val], clf.predict(X[val]), average=avg, zero_division=0))
    return np.mean(scores)

def tune_knn(X, y, n_folds=5):
    avg = 'macro' if len(np.unique(y)) > 2 else 'binary'
    best_score, best_p = -np.inf, {'n_neighbors': 5, 'metric': 'euclidean'}
    for k, m in iterproduct([1,3,5,7,9,11,13,15], ['euclidean','cityblock','chebyshev']):
        s = _cv_score(KNeighborsClassifier(n_neighbors=k, metric=m), X, y, n_folds, avg)
        if s > best_score: best_score, best_p = s, {'n_neighbors': k, 'metric': m}
    return best_p

def tune_parzen(X, y, n_folds=5):
    avg = 'macro' if len(np.unique(y)) > 2 else 'binary'
    best_score, best_h = -np.inf, 1.0
    for h in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]:
        s = _cv_score(ParzenWindowClassifier(h=h), X, y, n_folds, avg)
        if s > best_score: best_score, best_h = s, h
    return {'h': best_h}

def tune_logreg(X, y, n_folds=5):
    avg = 'macro' if len(np.unique(y)) > 2 else 'binary'
    nc = len(np.unique(y))
    mc = 'multinomial' if nc > 2 else 'auto'
    best_score, best_C = -np.inf, 1.0
    for C in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
        s = _cv_score(LogisticRegression(C=C, multi_class=mc, solver='lbfgs', max_iter=1000),
                      X, y, n_folds, avg)
        if s > best_score: best_score, best_C = s, C
    return {'C': best_C}

def build_classifiers(knn_p, par_p, lr_p, n_classes):
    mc = 'multinomial' if n_classes > 2 else 'auto'
    gb  = GaussianBayesClassifier()
    knn = KNeighborsClassifier(**knn_p)
    par = ParzenWindowClassifier(**par_p)
    lr  = LogisticRegression(**lr_p, multi_class=mc, solver='lbfgs', max_iter=1000)
    mv  = MajorityVoteClassifier([
        GaussianBayesClassifier(),
        KNeighborsClassifier(**knn_p),
        ParzenWindowClassifier(**par_p),
        LogisticRegression(**lr_p, multi_class=mc, solver='lbfgs', max_iter=1000)
    ])
    return {'GaussianBayes': gb, 'BayesKNN': knn, 'Parzen': par, 'LogReg': lr, 'MajorityVote': mv}


In [ ]:
def compute_metrics(y_true, y_pred, avg):
    return {
        'error_rate': 1 - accuracy_score(y_true, y_pred),
        'precision':  precision_score(y_true, y_pred, average=avg, zero_division=0),
        'recall':     recall_score(y_true, y_pred, average=avg, zero_division=0),
        'f1':         f1_score(y_true, y_pred, average=avg, zero_division=0),
    }


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EXPERIMENTO PRINCIPAL – 30 × 10-fold CV
# ═══════════════════════════════════════════════════════════════════════════

CLF_NAMES = ['GaussianBayes', 'BayesKNN', 'Parzen', 'LogReg', 'MajorityVote']
METRICS    = ['error_rate', 'precision', 'recall', 'f1']

def run_experiment(X, y, n_reps=30, n_outer=10, n_inner=5, rs_base=0, label=''):
    n_classes = len(np.unique(y))
    avg = 'macro' if n_classes > 2 else 'binary'

    # results[clf][metric] = list of per-rep means (length n_reps)
    results = {clf: {m: [] for m in METRICS} for clf in CLF_NAMES}

    for rep in range(n_reps):
        print(f"  [{label}] rep {rep+1:2d}/{n_reps}", end='\r')
        skf = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=rs_base + rep)

        fold_buf = {clf: {m: [] for m in METRICS} for clf in CLF_NAMES}

        for train_idx, test_idx in skf.split(X, y):
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            knn_p = tune_knn(X_tr, y_tr, n_inner)
            par_p = tune_parzen(X_tr, y_tr, n_inner)
            lr_p  = tune_logreg(X_tr, y_tr, n_inner)

            clfs = build_classifiers(knn_p, par_p, lr_p, n_classes)
            for name, clf in clfs.items():
                clf.fit(X_tr, y_tr)
                m = compute_metrics(y_te, clf.predict(X_te), avg)
                for k, v in m.items():
                    fold_buf[name][k].append(v)

        for clf in CLF_NAMES:
            for m in METRICS:
                results[clf][m].append(np.mean(fold_buf[clf][m]))

    print(f"\n  [{label}] concluído.")
    return results


print("Executando V1 (labels originais)...")
results_v1 = run_experiment(X, y_v1, n_reps=30, n_outer=10, n_inner=5, rs_base=0, label='V1')

print("Executando V2 (labels cluster c*=5)...")
results_v2 = run_experiment(X, y_v2, n_reps=30, n_outer=10, n_inner=5, rs_base=100, label='V2')

print("Experimento concluído.")


## Item b – Estimativas Pontuais e Intervalos de Confiança (t-Student 95%)

In [ ]:
def ci95(values):
    """Retorna (média, limite_inf, limite_sup) com IC t-Student 95%"""
    n = len(values)
    mu = np.mean(values)
    se = np.std(values, ddof=1) / np.sqrt(n)
    t_crit = t_dist.ppf(0.975, df=n-1)
    margin = t_crit * se
    return mu, mu - margin, mu + margin

print(f"{'Versão':<6} {'Métrica':<12} {'Classificador':<20} {'Média':>8} {'IC_inf':>10} {'IC_sup':>10}")
print("-" * 70)

for vname, res in [("V1", results_v1), ("V2", results_v2)]:
    for metric in METRICS:
        for clf in CLF_NAMES:
            mu, lo, hi = ci95(res[clf][metric])
            print(f"{vname:<6} {metric:<12} {clf:<20} {mu:>8.4f} {lo:>10.4f} {hi:>10.4f}")
        print()


In [ ]:
# Tabela resumida por métrica
for vname, res in [("V1", results_v1), ("V2", results_v2)]:
    print(f"\n=== {vname} ===")
    for metric in METRICS:
        rows = []
        for clf in CLF_NAMES:
            mu, lo, hi = ci95(res[clf][metric])
            rows.append({'Classificador': clf, 'Média': f"{mu:.4f}",
                         'IC 95%': f"[{lo:.4f}, {hi:.4f}]"})
        df = pd.DataFrame(rows)
        print(f"\n-- {metric} --")
        print(df.to_string(index=False))


## Item c – Teste de Friedman + Pós-teste de Nemenyi

In [ ]:
# Tabela q_α para Nemenyi (Demšar 2006, Tabela 5a)
Q_ALPHA = {2:{0.05:1.960,0.10:1.645}, 3:{0.05:2.343,0.10:2.052},
           4:{0.05:2.569,0.10:2.291}, 5:{0.05:2.728,0.10:2.459},
           6:{0.05:2.850,0.10:2.589}}

def friedman_nemenyi(results, metric, alpha=0.05):
    k = len(CLF_NAMES)
    N = len(results[CLF_NAMES[0]][metric])    # 30 repetições
    # Matriz de scores (N × k)
    scores = np.array([results[clf][metric] for clf in CLF_NAMES]).T

    # Ranks por linha (rank 1 = melhor)
    ranks = np.zeros_like(scores)
    reverse = (metric == 'error_rate')   # menor = melhor
    for i in range(N):
        row = scores[i]
        order = np.argsort(row) if reverse else np.argsort(row)[::-1]
        j = 0
        while j < k:
            j2 = j + 1
            while j2 < k and row[order[j2]] == row[order[j]]: j2 += 1
            avg_r = np.mean(np.arange(j+1, j2+1))
            for idx in range(j, j2): ranks[i, order[idx]] = avg_r
            j = j2

    R_j = ranks.mean(axis=0)   # rank médio por classificador

    # Estatística de Friedman (Iman-Davenport)
    chi2_F = 12*N / (k*(k+1)) * (np.sum(R_j**2) - k*(k+1)**2/4)
    F_F    = (N-1)*chi2_F / (N*(k-1) - chi2_F)
    p_val  = 1 - f_dist.cdf(F_F, k-1, (k-1)*(N-1))

    # Diferença crítica de Nemenyi
    q = Q_ALPHA[k][alpha]
    CD = q * np.sqrt(k*(k+1) / (6*N))

    return R_j, F_F, p_val, CD


print("Resultado do Teste de Friedman (Iman-Davenport)\n")
friedman_summary = {}

for vname, res in [("V1", results_v1), ("V2", results_v2)]:
    print(f"=== {vname} ===")
    friedman_summary[vname] = {}
    for metric in METRICS:
        R_j, F_F, p_val, CD = friedman_nemenyi(res, metric)
        sig = "*** SIGNIFICATIVO" if p_val < 0.05 else "(não significativo)"
        print(f"\n  {metric.upper():12s}: F_F={F_F:.4f}, p={p_val:.4f}  {sig}")
        print(f"  Ranks médios: " + ", ".join(f"{n}={r:.3f}" for n,r in zip(CLF_NAMES, R_j)))
        friedman_summary[vname][metric] = {'R_j': R_j, 'F_F': F_F, 'p': p_val, 'CD': CD}

        if p_val < 0.05:
            print(f"  CD (Nemenyi α=0.05) = {CD:.4f}")
            found = False
            for i in range(len(CLF_NAMES)):
                for j in range(i+1, len(CLF_NAMES)):
                    diff = abs(R_j[i] - R_j[j])
                    if diff > CD:
                        print(f"    DIFERENÇA: {CLF_NAMES[i]} vs {CLF_NAMES[j]}: "
                              f"|{R_j[i]:.3f}-{R_j[j]:.3f}|={diff:.3f} > CD={CD:.4f}")
                        found = True
            if not found:
                print("    (pós-teste não encontrou pares significativos)")
    print()


## Item d – Curvas de Aprendizado (F-measure)

In [ ]:
FRACTIONS = np.arange(0.05, 1.00, 0.05)   # 5% a 95%, passo 5%

def learning_curves(X, y, n_outer=10, n_inner=5, rs=0, label=''):
    n_classes = len(np.unique(y))
    avg = 'macro' if n_classes > 2 else 'binary'
    mc  = 'multinomial' if n_classes > 2 else 'auto'
    lc  = {clf: [] for clf in CLF_NAMES}

    skf = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=rs)
    splits = list(skf.split(X, y))

    for frac in FRACTIONS:
        fold_f1 = {clf: [] for clf in CLF_NAMES}
        for train_idx, test_idx in splits:
            X_tr, y_tr = X[train_idx], y[train_idx]
            X_te, y_te = X[test_idx],  y[test_idx]

            # Subamostra estratificada
            n_sub = max(n_classes * 2, int(np.round(frac * len(train_idx))))
            if n_sub >= len(train_idx):
                X_sub, y_sub = X_tr, y_tr
            else:
                try:
                    X_sub, _, y_sub, _ = train_test_split(
                        X_tr, y_tr, train_size=n_sub, stratify=y_tr, random_state=rs)
                except ValueError:
                    X_sub, y_sub = X_tr, y_tr

            if len(np.unique(y_sub)) < n_classes:
                for clf in CLF_NAMES: fold_f1[clf].append(np.nan)
                continue

            knn_p = tune_knn(X_sub, y_sub, min(n_inner, 3))
            par_p = tune_parzen(X_sub, y_sub, min(n_inner, 3))
            lr_p  = tune_logreg(X_sub, y_sub, min(n_inner, 3))
            clfs  = build_classifiers(knn_p, par_p, lr_p, n_classes)

            for name, clf in clfs.items():
                clf.fit(X_sub, y_sub)
                f1 = f1_score(y_te, clf.predict(X_te), average=avg, zero_division=0)
                fold_f1[name].append(f1)

        for clf in CLF_NAMES:
            vals = [v for v in fold_f1[clf] if not np.isnan(v)]
            lc[clf].append(np.mean(vals) if vals else np.nan)

        print(f"  [{label}] frac={frac:.2f} concluído", end='\r')

    print(f"\n  [{label}] curvas concluídas.")
    return lc


print("Calculando curvas de aprendizado V1...")
lc_v1 = learning_curves(X, y_v1, rs=200, label='V1')

print("Calculando curvas de aprendizado V2...")
lc_v2 = learning_curves(X, y_v2, rs=300, label='V2')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['tab:blue','tab:red','tab:green','tab:orange','tab:purple']
styles = ['-','--','-.',':', (0,(3,1,1,1))]

for ax, (vname, lc) in zip(axes, [("V1 – Labels Originais", lc_v1),
                                    (f"V2 – Clusters KCM-K-GH (c*=5)", lc_v2)]):
    for clf, col, ls in zip(CLF_NAMES, colors, styles):
        ax.plot(FRACTIONS * 100, lc[clf], label=clf, color=col,
                linestyle=ls, marker='o', markersize=4)
    ax.set_xlabel("Tamanho do treino (%)", fontsize=11)
    ax.set_ylabel("F-measure", fontsize=11)
    ax.set_title(f"Curvas de Aprendizado\n{vname}", fontsize=11)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([5, 95]); ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('curvas_aprendizado_q2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva: curvas_aprendizado_q2.png")


In [ ]:
# ── Diagrama de diferença crítica (CD) para F-measure ──────────────────
# Baseado em Demšar (2006), Figura 1

def plot_cd_diagram(R_j, CD, clf_names, title=''):
    fig, ax = plt.subplots(figsize=(8, 3))
    k = len(clf_names)
    # Ordenar por rank crescente (melhor à direita)
    order = np.argsort(R_j)
    ranked_names = [clf_names[i] for i in order]
    ranked_R     = R_j[order]

    ax.set_xlim([0.5, k + 0.5])
    ax.set_ylim([-1, 2])
    ax.set_yticks([])
    ax.set_xlabel('Rank médio (menor = melhor)', fontsize=10)
    ax.set_title(title, fontsize=10)

    # Eixo
    ax.axhline(1, color='black', lw=1.5)
    for i, (r, n) in enumerate(zip(ranked_R, ranked_names)):
        ax.plot(r, 1, 'ko', ms=6)
        ax.text(r, 1.15 if i % 2 == 0 else 0.75, n, ha='center', fontsize=8, rotation=30)

    # Barra CD
    best_r = ranked_R[0]
    ax.annotate('', xy=(best_r + CD, 1.5), xytext=(best_r, 1.5),
                arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
    ax.text((2*best_r + CD)/2, 1.62, f'CD={CD:.3f}', ha='center', color='red', fontsize=8)

    # Linhas de grupos não-significativos
    for i in range(k):
        for j in range(i+1, k):
            if abs(ranked_R[i] - ranked_R[j]) <= CD:
                y_bar = -0.3 - 0.15 * (i % 2)
                ax.plot([ranked_R[i], ranked_R[j]], [y_bar, y_bar], 'b-', lw=3, alpha=0.6)

    ax.invert_xaxis()
    plt.tight_layout()
    return fig

for vname, res in [("V1", results_v1), ("V2", results_v2)]:
    R_j, F_F, p_val, CD = friedman_nemenyi(res, 'f1')
    fig = plot_cd_diagram(R_j, CD, CLF_NAMES,
                          title=f'Diagrama CD – F-measure ({vname})  F_F={F_F:.3f} p={p_val:.4f}')
    fname = f'cd_diagram_{vname.lower()}_f1.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Salvo: {fname}")
